# Install Dependencies and Import

In [1]:
# Install Dependencies
%pip install -q --upgrade numerapi numerai-tools optuna seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.4 MB/s eta 0:00:00ta 0:00:01


In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import random
from datetime import timedelta
import time
import warnings
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
import gc
from functools import partial

# Numerai
from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

# Files
import json
import os
import pickle
import cloudpickle
import shutil
from tqdm import tqdm

# ML
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer

import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour
)

# Display Settings
pd.set_option("display.max_columns", 500)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Inline plots
%matplotlib inline

# Loading Numerai Datasets  

In [3]:
# Uploading Prepared Datasets
train = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/Step_1/train_medium.parquet")
val_neut = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/Step_1/val_neut.parquet")

with open('/content/drive/MyDrive/Colab Notebooks/Step_1/model_results.pkl', 'rb') as f:
    model_results = pickle.load(f)

params_final = model_results['tuned_cv_21']['params']
del model_results
gc.collect()

with open('/content/drive/MyDrive/Colab Notebooks/Step_1/grouped_set_medium.pkl', 'rb') as f:
    dict_medium = pickle.load(f) # Dict

feature_set = list(dict_medium['all'])

# for k, v in dict_medium.items():
#     print(k, len(v))

In [4]:
model_name = 'fm1'

#LGBMRegressor = partial(lgb.LGBMRegressor, verbosity=-1, device_type='gpu')

model = lgb.LGBMRegressor(**params_final, random_state=42, verbosity=-1, device_type='gpu')
model.fit(train[feature_set], train['target'])

with open(f'/content/drive/MyDrive/Colab Notebooks/Step_1/{model_name}.cpkl', 'wb') as f:
    cloudpickle.dump(model, f)

In [7]:
def compute_metrics_corr(df: pd.DataFrame, pred_col: str = 'prediction', target_col: str = 'target') -> dict:

    per_era_corr = df.groupby('era').apply(lambda g: numerai_corr(g[[pred_col]], g[target_col]))[pred_col]

    corr_mean = per_era_corr.mean() 
    corr_std = per_era_corr.std(ddof=0)
    corr_sharpe = corr_mean / corr_std
    corr_md = (per_era_corr.cumsum().expanding(min_periods=1).max() - per_era_corr.cumsum()).max()
    positive_eras = (per_era_corr > 0).mean()

    metrics_corr = {
        'corr_mean': corr_mean,
        'corr_std': corr_std,
        'corr_sharpe': corr_sharpe,
        'corr_md': corr_md,
        'positive_eras': positive_eras,
    }

    return metrics_corr


def compute_metrics_mmc(df: pd.DataFrame, pred_col: str = 'prediction', meta_col: str = 'meta_model', target_col: str = 'target') -> dict:

    df = df.dropna(subset=[pred_col, meta_col, target_col, 'era'])
    
    per_era_mmc = df.groupby('era').apply(lambda g: correlation_contribution(g[[pred_col]], g[meta_col], g[target_col]))[pred_col]

    mmc_mean = per_era_mmc.mean()
    mmc_std = per_era_mmc.std(ddof=0)
    mmc_sharpe = mmc_mean / mmc_std  # if mmc_std != 0 else 0.0
    mmc_md = ((per_era_mmc.cumsum().expanding(min_periods=1).max() - per_era_mmc.cumsum()).max())

    metrics_mmc = {
        'mmc_mean': mmc_mean,
        'mmc_std': mmc_std,
        'mmc_sharpe': mmc_sharpe,
        'mmc_md': mmc_md,
    }

    return metrics_mmc

# Межэровая автокорреляция (GPT)
def compute_autocorr_inter(df: pd.DataFrame, pred_col: str = 'prediction') -> float:
    
    pred_by_era = df.groupby('era')[pred_col].mean().sort_index()  # Агрегируем предсказания по эре
    corr = pred_by_era.corr(pred_by_era.shift(1), method='spearman')  # Сдвиг: era_t vs era_{t+1}  
    
    return round(corr, 4) if pd.notna(corr) else np.nan

# Внутриэровая автокорреляция
def compute_autocorr_intra(df: pd.DataFrame, pred_col: str = 'prediction') -> Dict[str, float]:
   
    def autocorr_era(group):
        return group[pred_col].corr(group[pred_col].shift(1))
    
    autocorrs = df.groupby('era', group_keys=True).apply(autocorr_era)
    autocorr_mean = autocorrs.mean()
    
    return autocorr_mean

def compute_feature_exposure(df: pd.DataFrame, pred_col: str, feature_set: List[str]) -> Dict[str, float]:

    X, y = df[feature_set], df[pred_col]
    
    return X.corrwith(y).abs().mean()

def get_neut_results(df: pd.DataFrame, pred_cols: List[str], feature_set: List[str])-> pd.DataFrame:

    results: List[Dict] = []

    for col in pred_cols:
    
        model_name = ('prediction' if col == 'prediction' else f'fnc_neut_{col.split("_")[1]}')

        metrics_corr = compute_metrics_corr(df, pred_col=col)
        metrics_mmc = compute_metrics_mmc(df, pred_col=col)
        autocorr_inter = compute_autocorr_inter(validation, pred_col=col)
        autocorr_intra = compute_autocorr_intra(validation, pred_col=col)
        feature_exposure = compute_feature_exposure(validation, pred_col=col, feature_set=feature_set) 

        result = {
            'model_name': model_name,
            'corr_mean': metrics_corr['corr_mean'],
            'corr_sharpe': metrics_corr['corr_sharpe'],
            'corr_std': metrics_corr['corr_std'],
            'corr_md': metrics_corr['corr_md'],
            'mmc_mean': metrics_mmc['mmc_mean'],
            'mmc_sharpe': metrics_mmc['mmc_sharpe'],
            'mmc_std': metrics_mmc['mmc_std'],
            'mmc_md': metrics_mmc['mmc_md'],
            'positive_eras': metrics_corr['positive_eras'],
            'autocorr_inter': autocorr_inter,
            'autocorr_intra': autocorr_intra,
            'fe': feature_exposure,
        }
        results.append(result)     
    
    return pd.DataFrame(results).round(4)

# metrics_corr = compute_metrics_corr(validation)
# metrics_mmc = compute_metrics_mmc(validation)
# autocorr_inter = compute_autocorr_inter(validation)
# autocorr_intra = compute_autocorr_intra(validation)
# fe = compute_feature_exposure(validation, pred_col='prediction', feature_set=feature_set)

# Groups for Neutralization

In [5]:
# 1. fncv3_features
fncv3_features = list(dict_medium['fncv3_features'])

# 2. Группы, которые мы хотим исключить
exclude_groups = {'intelligence', 'charisma', 'v2_equivalent_features', 'v3_equivalent_features'}

# 3. Собираем все признаки из остальных групп, кроме исключённых
other_features = []
for key, features in dict_medium.items():
    if key not in exclude_groups and key != 'fncv3_features' and key != 'all':
        other_features.extend(features)

fncv3_set = set(fncv3_features)
other_features = sorted(list(set(other_features)))
other_features = [f for f in other_features if f not in fncv3_set]

print("fncv3_features count:", len(fncv3_features))
print("other_features count:", len(other_features))

fncv3_features count: 400
other_features count: 372


# Neutralize FNCV3 Features

In [ ]:
# Neutralize predictions per-era against features at different proportions
proportions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for proportion in proportions:
    neutralized = validation.groupby('era', group_keys=True).apply(
        lambda d: neutralize(
          d[['prediction']],
          d[fncv3_features],
          proportion=proportion
        )
    ).reset_index().set_index('id')
    validation[f'neut_fnc_{proportion*100:.0f}'] = neutralized['prediction']

# Align the neutralized predictions with the validation data
# prediction_cols = ['prediction'] + [f for f in validation.columns if 'neut' in f]
# validation[['era', 'target'] + prediction_cols].head(3)

In [8]:
validation.to_parquet('/content/drive/MyDrive/Colab Notebooks/Step_1/val_neut.parquet')

In [9]:
prediction_cols = ['prediction'] + [f for f in validation.columns if 'neut_fnc' in f]

neut_results_df = get_neut_results(validation, pred_cols=prediction_cols, feature_set=feature_set)
display(neut_results_df)

,model_name,corr_mean,corr_sharpe,corr_std,corr_md,mmc_mean,mmc_sharpe,mmc_std,mmc_md,positive_eras,autocorr_inter,autocorr_intra,fe
0,prediction,0.0211,1.3677,0.0154,0.1219,-0.0037,-0.3112,0.0120,0.2448,0.9205,0.9713,-0.0009,0.0573
1,fnc_neut_fnc10,0.0214,1.3723,0.0156,0.1214,-0.0035,-0.2876,0.0122,0.2427,0.9156,0.9713,-0.0009,0.0542
2,fnc_neut_fnc20,0.0217,1.3755,0.0158,0.1210,-0.0033,-0.2627,0.0124,0.2401,0.9156,0.9713,-0.0009,0.0507
3,fnc_neut_fnc30,0.0219,1.3772,0.0159,0.1197,-0.0030,-0.2361,0.0127,0.2362,0.9123,0.9713,-0.0009,0.0468
4,fnc_neut_fnc40,0.0220,1.3782,0.0160,0.1177,-0.0027,-0.2080,0.0129,0.2305,0.9172,0.9713,-0.0009,0.0425
5,fnc_neut_fnc50,0.0221,1.3771,0.0161,0.1147,-0.0023,-0.1788,0.0131,0.2244,0.9205,0.9713,-0.0008,0.0379
6,fnc_neut_fnc60,0.0221,1.3722,0.0161,0.1115,-0.0020,-0.1494,0.0132,0.2167,0.9188,0.9713,-0.0008,0.0330
7,fnc_neut_fnc70,0.0220,1.3642,0.0161,0.1080,-0.0016,-0.1187,0.0134,0.2074,0.9156,0.9713,-0.0008,0.0279
8,fnc_neut_fnc80,0.0218,1.3534,0.0161,0.1036,-0.0012,-0.0856,0.0135,0.1961,0.9140,0.9713,-0.0007,0.0226
9,fnc_neut_fnc90,0.0215,1.3394,0.0160,0.0981,-0.0007,-0.0543,0.0135,0.1839,0.9091,0.9713,-0.0007,0.0172


In [10]:
neut_results_df.to_csv('/content/drive/MyDrive/Colab Notebooks/Step_1/neut_results.csv')

In [ ]:
# fig, axs = plt.subplots(2, 6, figsize=(18, 8))
# metrics = neut_results_df.columns[1:]
# df_plot = neut_results_df.set_index('model_name')

# for i, metric in enumerate(metrics):
#     ax = axs[i // 6, i % 6]
    
#     # Строим график с прозрачностью
#     df_plot[metric].plot(kind='bar', ax=ax, color='purple', width=0.8, alpha=0.8)
#     ax.set_title(metric, fontsize=10)

#     # Логарифмическая шкала (если применимо)
#     if metric not in ['mmc_mean', 'mmc_sharpe', 'autocorr_inter', 'autocorr_intra', 'positive_eras']:
#         ax.set_yscale('log')
#     else:

#         ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.7)

#     # Убираем метки
#     ax.set_xticks([])  # позиции по X
#     ax.set_xticklabels([])   
#     ax.tick_params(axis='both',which='both',labelleft=False,labelbottom=False,left=False,bottom=False)

#     ax.grid(True, axis='y', alpha=0.9, linewidth=0.7, color='black')

# plt.tight_layout()
# plt.show()


# Neutralize Other Features

In [15]:
pred_col = 'neut_fnc50'
proportions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


for proportion in proportions:

    neutralized = validation.groupby('era', group_keys=True).apply(
        lambda d: neutralize(
            d[[pred_col]].rename(columns={pred_col: 'prediction'}),  
            d[other_features],
            proportion=proportion
        )
    ).reset_index().set_index('id')
    validation[f'neut_other_{int(proportion*100)}'] = neutralized['prediction']

In [16]:
validation.to_parquet('/content/drive/MyDrive/Colab Notebooks/Step_1/val_neut.parquet')

Мы уже сделала нейтрализацию по двум группам и выбрали финальным предсказанием стоблбец neut_other_50
Как теперь загрузить модель на турнир Numerai, чтобы автоматически отправлялись предсказания во всех последующих раундах с приминенимем нейтрализации по двум группам с лучшей пропорцией.

In [17]:
prediction_cols = ['prediction'] + [pred_col] + [f for f in validation.columns if 'neut_other' in f]

neut_results_df = get_neut_results(validation, pred_cols=prediction_cols, feature_set=feature_set)
display(neut_results_df)

,model_name,corr_mean,corr_sharpe,corr_std,corr_md,mmc_mean,mmc_sharpe,mmc_std,mmc_md,positive_eras,autocorr_inter,autocorr_intra,fe
0,prediction,0.0211,1.3677,0.0154,0.1219,-0.0037,-0.3112,0.0120,0.2448,0.9205,0.9713,-0.0009,0.0573
1,fnc_neut_fnc50,0.0221,1.3771,0.0161,0.1147,-0.0023,-0.1788,0.0131,0.2244,0.9205,0.9713,-0.0008,0.0379
2,fnc_neut_other,0.0222,1.3780,0.0161,0.1133,-0.0022,-0.1695,0.0132,0.2225,0.9188,0.9713,-0.0008,0.0351
3,fnc_neut_other,0.0222,1.3773,0.0161,0.1111,-0.0021,-0.1584,0.0132,0.2201,0.9188,0.9713,-0.0008,0.0322
4,fnc_neut_other,0.0221,1.3728,0.0161,0.1100,-0.0020,-0.1484,0.0133,0.2175,0.9172,0.9713,-0.0007,0.0290
5,fnc_neut_other,0.0220,1.3652,0.0161,0.1082,-0.0018,-0.1372,0.0134,0.2133,0.9172,0.9713,-0.0007,0.0258
6,fnc_neut_other,0.0218,1.3538,0.0161,0.1062,-0.0017,-0.1246,0.0134,0.2091,0.9156,0.9713,-0.0006,0.0225
7,fnc_neut_other,0.0216,1.3414,0.0161,0.1022,-0.0015,-0.1113,0.0134,0.2029,0.9140,0.9713,-0.0006,0.0192
8,fnc_neut_other,0.0213,1.3239,0.0161,0.0996,-0.0013,-0.0976,0.0134,0.1969,0.9107,0.9713,-0.0005,0.0159
9,fnc_neut_other,0.0209,1.3036,0.0160,0.0969,-0.0011,-0.0839,0.0133,0.1902,0.9091,0.9713,-0.0005,0.0129


In [18]:
# Initialize NumerAPI
napi = NumerAPI()

# Get current active round for Numerai Tournament
current_round = napi.get_current_round()
print("CURRENT ROUND:", current_round)
print('')

# List all available datasets
all_datasets = napi.list_datasets()

# Extract unique data versions
dataset_versions = list(set(d.split('/')[0] for d in all_datasets))
DATA_VERSION = max(dataset_versions)  # Use latest version
print("AVAILABLE DATA VERSIONS:\n", sorted(dataset_versions))
print('')

# List all files from the latest data version
print(f"FILES IN VERSION {DATA_VERSION}:")
current_version_files = [f for f in all_datasets if f.startswith(DATA_VERSION)]
for f in sorted(current_version_files):
    print(f"  {f}")

CURRENT ROUND: 1168

AVAILABLE DATA VERSIONS:
 ['v5.0', 'v5.1', 'v5.2']

FILES IN VERSION v5.2:
  v5.2/features.json
  v5.2/live.parquet
  v5.2/live_benchmark_models.parquet
  v5.2/live_example_preds.csv
  v5.2/live_example_preds.parquet
  v5.2/meta_model.parquet
  v5.2/train.parquet
  v5.2/train_benchmark_models.parquet
  v5.2/validation.parquet
  v5.2/validation_benchmark_models.parquet
  v5.2/validation_example_preds.csv
  v5.2/validation_example_preds.parquet


In [ ]:
napi.download_dataset(f"{DATA_VERSION}/live.parquet")
live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_set)

In [ ]:
# Download latest live features
napi.download_dataset(f"{DATA_VERSION}/live.parquet")

# Load live features
live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_set)

# Generate live predictions
live_predictions = model.predict(live_features[feature_set])

# Format submission
pd.Series(live_predictions, index=live_features.index).to_frame("prediction")

# Define your prediction pipeline as a function
def predict(live_features: pd.DataFrame, _live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    live_predictions = model.predict(live_features[feature_set])
    submission = pd.Series(live_predictions, index=live_features.index)
    return submission.to_frame("prediction")

# Use the cloudpickle library to serialize your function
import cloudpickle
p = cloudpickle.dumps(predict)
with open("hello_numerai.pkl", "wb") as f:
    f.write(p)

In [ ]:
def predict_neutral(live_features: pd.DataFrame, _live_benchmark_models: pd.DataFrame = None) -> pd.DataFrame:
    # make predictions using all features
    predictions = pd.DataFrame(
        model.predict(live_features[small_features]),
        index=live_features.index,
        columns=["prediction"]
    )
    # neutralize predictions to a subset of features
    neutralized = neutralize(predictions, live_features[sm_serenity_feats])
    return neutralized.rank(pct=True)
